In [1]:
import sys
print(sys.executable)
print(sys.version)

/opt/anaconda3/envs/data1030/bin/python
3.14.6 | packaged by conda-forge | (main, Aug 11 2026, 10:24:52) [Clang 20.1.8 ]


In [2]:

import pandas as pd
from pathlib import Path

# Find project root
ROOT = Path.cwd().resolve()

while not (ROOT / "data").exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Project data folder not found")
    ROOT = ROOT.parent

# Load cleaned data
data_path = ROOT / "data" / "processed" / "complaints_clean.csv"

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head())


Dataset shape: (9461, 15)

Columns:
['complaint_id', 'make', 'model', 'model_year', 'product_model', 'filed_date', 'incident_date', 'components', 'crash', 'fire', 'number_of_injuries', 'number_of_deaths', 'summary', 'reporting_delay', 'invalid_incident_date']


,complaint_id,make,model,model_year,product_model,filed_date,incident_date,components,crash,fire,number_of_injuries,number_of_deaths,summary,reporting_delay,invalid_incident_date
0,11770133,TOYOTA,CAMRY,2018,CAMRY,2026-10-09,2026-09-29,POWER TRAIN,False,False,0,0,"Total transmission failure at about 107,770 mi...",10.0,False
1,11769969,TOYOTA,CAMRY,2018,CAMRY,2026-10-08,2026-10-06,ENGINE,False,False,0,0,My 2018 Toyota Camry LE went into Limp mode wh...,2.0,False
2,11769124,TOYOTA,CAMRY,2018,CAMRY,2026-10-05,2026-10-03,POWER TRAIN,False,False,0,0,The contact owns a 2018 Toyota Camry. The cont...,2.0,False
3,11768868,TOYOTA,CAMRY,2018,CAMRY,2026-10-04,2026-10-02,FUEL/PROPULSION SYSTEM,False,False,0,0,I am filing a formal complaint regarding a sev...,2.0,False
4,11768335,TOYOTA,CAMRY,2018,CAMRY,2026-10-01,2026-09-29,POWER TRAIN,False,False,0,0,"2018 Toyota Camry approx 126,000 miles. Trans...",2.0,False


In [3]:

import zipfile

# Paths to the new investigation files
investigation_dir = ROOT / "data" / "raw" / "Investigation"

zip_path = investigation_dir / "FLAT_INV.zip"
dictionary_path = investigation_dir / "INV.txt"

print("ZIP exists:", zip_path.exists())
print("Dictionary exists:", dictionary_path.exists())

# List files inside ZIP
with zipfile.ZipFile(zip_path, "r") as z:
    for file in z.namelist():
        print("File inside ZIP:", file)


ZIP exists: True
Dictionary exists: True
File inside ZIP: FLAT_INV.txt


In [4]:

# Read NHTSA investigation data dictionary

with open(
    dictionary_path,
    "r",
    encoding="utf-8-sig",
    errors="replace"
) as f:
    dictionary_text = f.read()

print(dictionary_text)



DEFECT INVESTIGATIONS

The INVESTIGATION file contains all NHTSA safety-related defect investigations 
opened since 1972.

File characteristics:

-  All the records are TAB delimited
-  All dates are in YYYYMMDD format

-  Maximum Record length: 6816 characters

Change log:
1.	September 14, 2007
	-Changed flat file extension from .lst to .txt
2.	March 14, 2008
	-Field #11 added
3.	July 31, 2023
	-Changed Field #10 length from 40 to 200
	-Changed Field #11 length from 4000 to 6000
	-Investigations are now sorted by NHTSA Identification Number (Field #1)
4.	September 5, 2023
	-Investigations are now sorted by NHTSA Identification Number alphabetically (Field #1)


Last Updated September 5, 2023


FIELDS:

    Note: The CAMPNO field (NHTSA Recall Campaign Number), if present, 
          identifies the recall campaign initiated as a result of the 
          investigation.  The CAMPNO can be used to link to the RECALLS 
          database for additional information.  
          

Field#   

In [5]:

import pandas as pd
import zipfile

# Column names from the official NHTSA data dictionary
inv_columns = [
    "action_number",
    "make",
    "model",
    "model_year",
    "component",
    "manufacturer",
    "open_date",
    "close_date",
    "campaign_number",
    "subject",
    "summary"
]

# Inspect ZIP contents
with zipfile.ZipFile(zip_path, "r") as z:
    files = [
        name for name in z.namelist()
        if name.lower().endswith(".txt")
    ]

    print("Text files inside ZIP:", files)

    if len(files) != 1:
        raise ValueError(
            "Please inspect ZIP contents before choosing the data file."
        )

    with z.open(files[0]) as f:
        inv_df = pd.read_csv(
            f,
            sep="\t",
            header=None,
            names=inv_columns,
            dtype=str,
            encoding="latin-1",
            on_bad_lines="error"
        )

print("Investigation dataset shape:", inv_df.shape)

display(inv_df.head())


Text files inside ZIP: ['FLAT_INV.txt']
Investigation dataset shape: (154502, 11)


,action_number,make,model,model_year,component,manufacturer,open_date,close_date,campaign_number,subject,summary
0,AQ08001,PACE AMERICAN,TRAILER,2003,WHEELS,"PACE AMERICAN, INC.",20080618,20081029,NaN,PACE AMERICAN 573 RETRACTION,"BY LETTER DATED NOVEMBER 8, 2007, PACE AMERICA..."
1,AQ08001,PACE AMERICAN,TRAILER,2004,WHEELS,"PACE AMERICAN, INC.",20080618,20081029,NaN,PACE AMERICAN 573 RETRACTION,"BY LETTER DATED NOVEMBER 8, 2007, PACE AMERICA..."
2,AQ08001,PACE AMERICAN,TRAILER,2005,WHEELS,"PACE AMERICAN, INC.",20080618,20081029,NaN,PACE AMERICAN 573 RETRACTION,"BY LETTER DATED NOVEMBER 8, 2007, PACE AMERICA..."
3,AQ08001,PACE AMERICAN,TRAILER,2006,WHEELS,"PACE AMERICAN, INC.",20080618,20081029,NaN,PACE AMERICAN 573 RETRACTION,"BY LETTER DATED NOVEMBER 8, 2007, PACE AMERICA..."
4,AQ09001,APEXCONE,APX9004,9999,EXTERIOR LIGHTING,CARFILO,20090326,20090706,05E069000,HID REPLACEMENT KIT RECALL CAMPAIGNS,RMD IDENTIFIED SEVERAL HID REPLACEMENT LIGHTIN...


In [6]:

# Basic inspection of investigation records

print("Total rows:", len(inv_df))

print(
    "Unique investigations:",
    inv_df["action_number"].nunique()
)

print("\nMissing values:")
display(inv_df.isna().sum())

print("\nInvestigation action numbers:")
display(inv_df["action_number"].value_counts().head(10))

print("\nVehicle makes:")
display(inv_df["make"].value_counts().head(15))


Total rows: 154502
Unique investigations: 5359

Missing values:


action_number          0
make                1358
model               1358
model_year          1358
component           1363
manufacturer        1358
open_date            157
close_date         75583
campaign_number    28741
subject                0
summary              556
dtype: int64


Investigation action numbers:


action_number
EA15001    71004
PE19004     7840
DP18003     7072
PE19003     6048
PE14016     3294
EA21002     2776
EA05005     2037
EQ08011     1890
AQ09001     1120
PE21003      940
Name: count, dtype: int64


Vehicle makes:


make
HONDA        16625
DODGE        13462
KIA          12796
FORD         10433
BMW           9882
HYUNDAI       9836
MAZDA         7306
TOYOTA        6332
CHRYSLER      4908
SUBARU        4754
INFINITI      4333
CHEVROLET     3951
ACURA         3553
NISSAN        3344
GMC           2433
Name: count, dtype: int64

In [7]:

# A2.5 - Get the original 10 vehicle model families

model_families = (
    df[["make", "model"]]
    .drop_duplicates()
    .sort_values(["make", "model"])
    .reset_index(drop=True)
)

print("Number of model families:", len(model_families))

display(model_families)


Number of model families: 10


,make,model
0,AUDI,Q5
1,BMW,3 SERIES
2,FORD,ESCAPE
3,FORD,F-150
4,HONDA,CIVIC
5,HYUNDAI,ELANTRA
6,MERCEDES-BENZ,C-CLASS
7,NISSAN,ALTIMA
8,TOYOTA,CAMRY
9,VOLKSWAGEN,TIGUAN


In [8]:

# A2.6 - Expand model years from 2015 to 2023

years = pd.DataFrame({
    "model_year": range(2015, 2024)
})

expanded_groups = model_families.merge(
    years,
    how="cross"
)

print("Number of expanded vehicle groups:", len(expanded_groups))

display(expanded_groups.head(20))


Number of expanded vehicle groups: 90


,make,model,model_year
0,AUDI,Q5,2015
1,AUDI,Q5,2016
2,AUDI,Q5,2017
3,AUDI,Q5,2018
4,AUDI,Q5,2019
5,AUDI,Q5,2020
6,AUDI,Q5,2021
7,AUDI,Q5,2022
8,AUDI,Q5,2023
9,BMW,3 SERIES,2015


In [9]:

# A2.7 - Clean matching keys and dates

inv_clean = inv_df.copy()

for col in ["make", "model"]:
    inv_clean[col] = (
        inv_clean[col]
        .astype("string")
        .str.strip()
        .str.upper()
    )

inv_clean["model_year"] = pd.to_numeric(
    inv_clean["model_year"],
    errors="coerce"
)

inv_clean["open_date"] = pd.to_datetime(
    inv_clean["open_date"],
    format="%Y%m%d",
    errors="coerce"
)

# Standardize keys in our expanded vehicle groups
for col in ["make", "model"]:
    expanded_groups[col] = (
        expanded_groups[col]
        .astype("string")
        .str.strip()
        .str.upper()
    )

print("Investigation rows:", len(inv_clean))
print("Valid opening dates:", inv_clean["open_date"].notna().sum())
print("Missing opening dates:", inv_clean["open_date"].isna().sum())


Investigation rows: 154502
Valid opening dates: 154345
Missing opening dates: 157


In [10]:

# A2.8 - Exact matching with expanded vehicle groups

matching_keys = ["make", "model", "model_year"]

inv_matched = inv_clean.merge(
    expanded_groups,
    on=matching_keys,
    how="inner",
    validate="many_to_one"
)

print("Matched investigation rows:", len(inv_matched))

print(
    "Unique matched action numbers:",
    inv_matched["action_number"].nunique()
)

print(
    "Matched vehicle groups:",
    inv_matched[matching_keys].drop_duplicates().shape[0]
)

display(
    inv_matched[
        [
            "action_number",
            "make",
            "model",
            "model_year",
            "open_date",
            "component"
        ]
    ].head(20)
)


Matched investigation rows: 108
Unique matched action numbers: 29
Matched vehicle groups: 29


,action_number,make,model,model_year,open_date,component
0,AQ17001,FORD,F-150,2015.0,2017-01-04,STEERING
1,AQ22002,FORD,ESCAPE,2021.0,2022-06-03,"FUEL SYSTEM, GASOLINE:DELIVERY:FUEL PUMP:CONTR..."
2,AQ22003,NISSAN,ALTIMA,2018.0,2022-12-16,LATCHES/LOCKS/LINKAGES:HOOD:LATCH
3,AQ22003,NISSAN,ALTIMA,2019.0,2022-12-16,LATCHES/LOCKS/LINKAGES:HOOD:LATCH
4,AQ22003,NISSAN,ALTIMA,2020.0,2022-12-16,LATCHES/LOCKS/LINKAGES:HOOD:LATCH
5,AQ22003,NISSAN,ALTIMA,2021.0,2022-12-16,LATCHES/LOCKS/LINKAGES:HOOD:LATCH
6,AQ22003,NISSAN,ALTIMA,2022.0,2022-12-16,LATCHES/LOCKS/LINKAGES:HOOD:LATCH
7,AQ23002,HYUNDAI,ELANTRA,2015.0,2023-11-17,"SERVICE BRAKES, HYDRAULIC:ANTILOCK/TRACTION CO..."
8,DP21003,HYUNDAI,ELANTRA,2015.0,2021-11-12,VEHICLE SPEED CONTROL:THROTTLE
9,DP21003,HYUNDAI,ELANTRA,2016.0,2021-11-12,VEHICLE SPEED CONTROL:THROTTLE


In [11]:

# A2.9 - Summarize matches by vehicle family

match_summary = (
    inv_matched
    .groupby(["make", "model"])
    .agg(
        matched_rows=("action_number", "size"),
        unique_actions=("action_number", "nunique"),
        model_years_matched=("model_year", "nunique")
    )
    .reset_index()
)

# Include families with zero exact matches
match_summary = model_families.merge(
    match_summary,
    on=["make", "model"],
    how="left"
)

count_cols = [
    "matched_rows",
    "unique_actions",
    "model_years_matched"
]

match_summary[count_cols] = (
    match_summary[count_cols].fillna(0).astype(int)
)

display(
    match_summary.sort_values(
        "unique_actions",
        ascending=False
    )
)


,make,model,matched_rows,unique_actions,model_years_matched
3,FORD,F-150,27,9,8
2,FORD,ESCAPE,21,6,4
5,HYUNDAI,ELANTRA,19,5,5
7,NISSAN,ALTIMA,30,5,8
4,HONDA,CIVIC,7,4,3
9,VOLKSWAGEN,TIGUAN,4,2,1
0,AUDI,Q5,0,0,0
1,BMW,3 SERIES,0,0,0
6,MERCEDES-BENZ,C-CLASS,0,0,0
8,TOYOTA,CAMRY,0,0,0


In [12]:


# A2.10 - Correct investigation summary

inv_review = inv_matched.copy()

# Create a unique key for each vehicle model-year group
inv_review["vehicle_group"] = (
    inv_review["make"].astype(str) + "|" +
    inv_review["model"].astype(str) + "|" +
    inv_review["model_year"].astype(str)
)

investigation_summary = (
    inv_review.groupby("action_number")
    .agg(
        open_date=("open_date", "min"),
        matched_rows=("action_number", "size"),
        unique_vehicle_groups=("vehicle_group", "nunique"),
        makes=("make", lambda x: ", ".join(sorted(x.dropna().unique()))),
        models=("model", lambda x: ", ".join(sorted(x.dropna().unique())))
    )
    .reset_index()
    .sort_values("open_date")
)

display(investigation_summary)

print("Unique action numbers:", len(investigation_summary))



,action_number,open_date,matched_rows,unique_vehicle_groups,makes,models
14,PE16001,2016-01-16,2,1,FORD,ESCAPE
24,RQ16001,2016-09-28,8,2,FORD,F-150
0,AQ17001,2017-01-04,1,1,FORD,F-150
15,PE18001,2018-01-10,1,1,VOLKSWAGEN,TIGUAN
6,EA18005,2018-07-03,3,1,VOLKSWAGEN,TIGUAN
16,PE18008,2018-08-03,4,4,FORD,F-150
17,PE18013,2018-12-01,4,4,NISSAN,ALTIMA
7,EA19001,2019-04-19,1,1,HONDA,CIVIC
8,EA19002,2019-07-29,4,4,NISSAN,ALTIMA
25,RQ19005,2019-12-19,8,1,FORD,ESCAPE


Unique action numbers: 29


In [13]:

# A2.11 - Count unique action numbers by opening year

investigation_summary["open_year"] = (
    investigation_summary["open_date"].dt.year
)

year_summary = (
    investigation_summary
    .groupby("open_year", dropna=False)
    .agg(
        unique_actions=("action_number", "nunique")
    )
    .reset_index()
    .sort_values("open_year")
)

display(year_summary)


,open_year,unique_actions
0,2016,2
1,2017,1
2,2018,4
3,2019,3
4,2021,3
5,2022,4
6,2023,7
7,2024,2
8,2025,1
9,2026,2


In [14]:

# A2.12 - Inspect action number prefixes

investigation_summary["action_type"] = (
    investigation_summary["action_number"].str[:2]
)

type_summary = (
    investigation_summary
    .groupby("action_type")
    .agg(
        unique_actions=("action_number", "nunique")
    )
    .reset_index()
    .sort_values("unique_actions", ascending=False)
)

display(type_summary)


,action_type,unique_actions
3,PE,10
2,EA,8
4,RQ,5
0,AQ,4
1,DP,2


In [15]:

# A2.13 - Inspect possible model naming differences

unmatched_makes = [
    "AUDI",
    "BMW",
    "MERCEDES-BENZ",
    "TOYOTA"
]

for make in unmatched_makes:
    models = (
        inv_clean.loc[
            (inv_clean["make"] == make) &
            (inv_clean["model_year"].between(2015, 2023)),
            "model"
        ]
        .dropna()
        .value_counts()
    )

    print(f"\n===== {make} =====")
    display(models.head(50))



===== AUDI =====


model
A3              18
S3              18
A3 CABRIOLET    15
RS 3             9
A8               8
RS7              8
S6               8
S7               8
S8               8
E-TRON           4
A3 ETRON         3
Q5 PHEV          3
A7 PHEV          2
A8 PHEV          2
Q5 HYBRID        2
Name: count, dtype: Int64


===== BMW =====


model
X5                    18
I8                    16
328D                  12
330E                  11
X6                    10
640I                  10
X4                     9
X3                     9
535I                   8
I3                     8
K1600B                 8
320I                   8
650I                   8
X6 M                   8
335I                   7
530E                   7
X1                     7
540I                   6
340I                   6
430I                   6
440I                   6
530I                   6
535D                   6
550I                   6
X1 SAV                 6
X2                     5
R1250 GS ADVENTURE     5
R 1250 RT              5
228I                   4
230I                   4
320XI                  4
328I                   4
328XD                  4
328XI                  4
330I                   4
335XI                  4
428I                   4
428XI                  4
435I                   4
528I               


===== MERCEDES-BENZ =====


model
SPRINTER 2500      14
SPRINTER 3500      14
C300               12
AMG CLA45          12
CLA250             11
E350               10
C350E              10
AMG GLC43           9
GLC300              9
GLA 250             8
AMG GT              6
C350                6
E400                6
SL550               6
SPRINTER 1500       6
SPRINTER 4500       6
AMG S63             6
GLA250              6
C450                5
GLC350E             4
S550                4
AMG C43             4
AMG C63             4
AMG C63S            4
AMG CLS63           4
AMG SLC43           4
B160                4
B 250E              4
C250                4
C 300               4
CLS550              4
CLS 550             4
E250                4
E300                4
E63                 4
GLC                 4
SL400               4
SL63                4
SLC300              4
SLC 43 AMG          4
SLK250              4
SLK300              4
SLK350              4
AMG GLE63           4
SPRINTER (VS30)     4
GLE5


===== TOYOTA =====


model
AVALON HYBRID           15
COROLLA                 13
PRIUS                   11
CAMRY HYBRID             8
HIGHLANDER HYBRID        8
PRIUS PRIME              6
RAV4 HYBRID              6
4RUNNER                  6
PRIUS C                  5
TACOMA                   5
AVALON                   4
COROLLA IM               4
RAV4                     4
PRIUS PLUG-IN HYBRID     3
COROLLA HYBRID           3
SEQUOIA                  3
TUNDRA                   3
RAV4 PRIME               2
SIENNA HYBRID            2
VENZA HYBRID             2
YARIS                    2
RAV4 HV                  1
TUNDRA HYBRID            1
Name: count, dtype: Int64

In [16]:

# A2.14 - Review possible model aliases

alias_candidates = {
    "AUDI": [
        "Q5 PHEV", "Q5 HYBRID"
    ],
    "BMW": [
        "320I", "320XI", "328I", "328XI",
        "328D", "328XD", "330I", "330E",
        "335I", "335XI", "340I",
        "ACTIVEHYBRID 3", "M340I"
    ],
    "MERCEDES-BENZ": [
        "C250", "C300", "C 300", "C350",
        "C350E", "C400", "C450",
        "AMG C43", "AMG C63", "AMG C63S"
    ],
    "TOYOTA": [
        "CAMRY HYBRID"
    ]
}

candidate_frames = []

for make, models in alias_candidates.items():
    subset = inv_clean.loc[
        (inv_clean["make"] == make) &
        (inv_clean["model"].isin(models)) &
        (inv_clean["model_year"].between(2015, 2023))
    ].copy()

    candidate_frames.append(subset)

alias_review = pd.concat(
    candidate_frames,
    ignore_index=True
)

alias_review = (
    alias_review[
        [
            "action_number", "make", "model",
            "model_year", "open_date", "subject"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["make", "action_number", "model_year"]
    )
)

print("Candidate records:", len(alias_review))
print(
    "Unique candidate action numbers:",
    alias_review["action_number"].nunique()
)

display(alias_review)


Candidate records: 91
Unique candidate action numbers: 5


,action_number,make,model,model_year,open_date,subject
0,DP22005,AUDI,Q5 HYBRID,2015.0,2023-01-27,Pedestrian alert sounds
1,DP22005,AUDI,Q5 HYBRID,2016.0,2023-01-27,Pedestrian alert sounds
2,DP22005,AUDI,Q5 PHEV,2020.0,2023-01-27,Pedestrian alert sounds
3,DP22005,AUDI,Q5 PHEV,2021.0,2023-01-27,Pedestrian alert sounds
4,DP22005,AUDI,Q5 PHEV,2022.0,2023-01-27,Pedestrian alert sounds
...,...,...,...,...,...,...
136,DP22005,TOYOTA,CAMRY HYBRID,2018.0,2023-01-27,Pedestrian alert sounds
137,DP22005,TOYOTA,CAMRY HYBRID,2019.0,2023-01-27,Pedestrian alert sounds
138,DP22005,TOYOTA,CAMRY HYBRID,2020.0,2023-01-27,Pedestrian alert sounds
139,DP22005,TOYOTA,CAMRY HYBRID,2021.0,2023-01-27,Pedestrian alert sounds


In [17]:

# A2.17 - Standardize investigation model families

inv_clean["model_family"] = inv_clean["model"].copy()

model_aliases = {
    "AUDI": {
        "Q5 HYBRID": "Q5",
        "Q5 PHEV": "Q5"
    },
    "BMW": {
        "320I": "3 SERIES",
        "320XI": "3 SERIES",
        "328I": "3 SERIES",
        "328XI": "3 SERIES",
        "328D": "3 SERIES",
        "328XD": "3 SERIES",
        "330I": "3 SERIES",
        "330E": "3 SERIES",
        "335I": "3 SERIES",
        "335XI": "3 SERIES",
        "340I": "3 SERIES",
        "ACTIVEHYBRID 3": "3 SERIES",
        "M340I": "3 SERIES"
    },
    "MERCEDES-BENZ": {
        "C250": "C-CLASS",
        "C300": "C-CLASS",
        "C 300": "C-CLASS",
        "C350": "C-CLASS",
        "C350E": "C-CLASS",
        "C400": "C-CLASS",
        "C450": "C-CLASS",
        "AMG C43": "C-CLASS",
        "AMG C63": "C-CLASS",
        "AMG C63S": "C-CLASS"
    },
    "TOYOTA": {
        "CAMRY HYBRID": "CAMRY"
    }
}

for make, mapping in model_aliases.items():
    mask = inv_clean["make"] == make

    inv_clean.loc[mask, "model_family"] = (
        inv_clean.loc[mask, "model"]
        .replace(mapping)
    )

print("Model family mapping completed.")


Model family mapping completed.


In [18]:

# A2.18 - Re-match investigations with expanded vehicle groups

expanded_match = expanded_groups.rename(
    columns={"model": "model_family"}
)

matching_keys = ["make", "model_family", "model_year"]

inv_matched_final = inv_clean.merge(
    expanded_match,
    on=matching_keys,
    how="inner",
    validate="many_to_one"
)

print("Matched investigation rows:", len(inv_matched_final))
print("Unique action numbers:",
      inv_matched_final["action_number"].nunique())
print("Matched vehicle groups:",
      inv_matched_final[matching_keys].drop_duplicates().shape[0])


Matched investigation rows: 249
Unique action numbers: 33
Matched vehicle groups: 54


In [19]:

# A2.19 - Summarize matching results

final_summary = (
    inv_matched_final
    .groupby(["make", "model_family"])
    .agg(
        matched_rows=("action_number", "size"),
        unique_actions=("action_number", "nunique"),
        model_years_matched=("model_year", "nunique")
    )
    .reset_index()
)

all_families = (
    expanded_match[["make", "model_family"]]
    .drop_duplicates()
)

final_summary = all_families.merge(
    final_summary,
    on=["make", "model_family"],
    how="left"
)

count_cols = [
    "matched_rows",
    "unique_actions",
    "model_years_matched"
]

final_summary[count_cols] = (
    final_summary[count_cols].fillna(0).astype(int)
)

display(
    final_summary.sort_values(
        "unique_actions", ascending=False
    )
)


,make,model_family,matched_rows,unique_actions,model_years_matched
3,FORD,F-150,27,9,8
2,FORD,ESCAPE,21,6,4
5,HYUNDAI,ELANTRA,19,5,5
7,NISSAN,ALTIMA,30,5,8
4,HONDA,CIVIC,7,4,3
6,MERCEDES-BENZ,C-CLASS,56,4,6
1,BMW,3 SERIES,72,3,6
9,VOLKSWAGEN,TIGUAN,4,2,1
0,AUDI,Q5,5,1,5
8,TOYOTA,CAMRY,8,1,8


In [20]:

# A3.1 - Inspect available data files

from pathlib import Path

print("Project root:", ROOT)

for folder in ["raw", "processed"]:
    folder_path = ROOT / "data" / folder

    print(f"\n===== {folder.upper()} DATA =====")

    if folder_path.exists():
        for file in sorted(folder_path.rglob("*")):
            if file.is_file():
                size_mb = file.stat().st_size / (1024 ** 2)

                print(
                    f"{file.relative_to(ROOT)} "
                    f"({size_mb:.2f} MB)"
                )
    else:
        print("Folder not found")


Project root: /Users/xinyisong/Desktop/DATA1030-Midterm-Xinfinity

===== RAW DATA =====
data/raw/.gitkeep (0.00 MB)
data/raw/Complaints/CMPL.txt (0.01 MB)
data/raw/Complaints/FLAT_CMPL.zip (359.51 MB)
data/raw/Investigation/FLAT_INV.zip (4.11 MB)
data/raw/Investigation/INV.txt (0.00 MB)

===== PROCESSED DATA =====
data/processed/complaints_clean.csv (6.22 MB)
data/processed/complaints_final.csv (26.34 MB)
data/processed/modeling_dataset.csv (0.29 MB)


In [21]:

# A3.3 - Inspect raw NHTSA complaints data

import zipfile
import pandas as pd

complaints_zip = (
    ROOT / "data" / "raw" / "Complaints" / "FLAT_CMPL.zip"
)

with zipfile.ZipFile(complaints_zip, "r") as z:
    print("Files inside ZIP:")
    print(z.namelist())

    txt_files = [
        name for name in z.namelist()
        if name.lower().endswith(".txt")
    ]

    with z.open(txt_files[0]) as f:
        sample = pd.read_csv(
            f,
            sep="\t",
            header=None,
            nrows=5,
            dtype=str,
            encoding="latin-1"
        )

print("\nSample shape:", sample.shape)
print("Number of columns:", sample.shape[1])

display(sample)


Files inside ZIP:
['FLAT_CMPL.txt']

Sample shape: (5, 51)
Number of columns: 51


,0,1,2,3,4,5,6,7,8,9,...,41,42,43,44,45,46,47,48,49,50
0,1,958241,"Volvo Car USA, LLC",VOLVO,760,1987,N,NaN,N,0,...,NaN,NaN,NaN,NaN,V,NaN,N,N,NaN,NaN
1,2,958130,Ford Motor Company,FORD,THUNDERBIRD,1992,N,19941222,N,0,...,NaN,NaN,NaN,NaN,V,NaN,N,N,NaN,NaN
2,3,958132,"Kia America, Inc.",KIA,SEPHIA,1994,Y,19941230,N,0,...,NaN,NaN,NaN,NaN,V,NaN,N,N,NaN,NaN
3,4,958133,"Chrysler (FCA US, LLC)",DODGE,600,1987,N,19941231,N,0,...,NaN,NaN,NaN,NaN,V,NaN,N,N,NaN,NaN
4,5,958137,"Chrysler (FCA US, LLC)",DODGE,CARAVAN,1991,N,19941218,N,0,...,NaN,NaN,NaN,NaN,V,NaN,N,N,NaN,NaN


In [22]:

# A3.4 - Read official complaints data dictionary

dictionary_path = (
    ROOT / "data" / "raw" / "Complaints" / "CMPL.txt"
)

with open(dictionary_path, "r", encoding="latin-1") as f:
    dictionary_text = f.read()

print(dictionary_text[:15000])


CONSUMER COMPLAINTS

The COMPLAINTS file contains all safety-related defect complaints received 
by NHTSA since January 1, 1995.

File characteristics:

-  All the records are TAB delimited
-  All dates are in YYYYMMDD format

-  Maximum record length: 2824

Change log:
1. Fields 21 - 46 were added on Sept. 14, 2007
2. Changed flat file extension from .lst to .txt on Sept. 14, 2007
3. Field 47 was added on Oct. 15, 2007
4. Field 34 was changed from CHAR(20) to CHAR(30) on Aug. 08, 2008
5. Field 18 was changed from NUMBER(6) to NUMBER(7) on Jun. 18, 2010
6. Complaint Type 'MIVQ' was added to Field 21 list on Mar. 21, 2013
7. Complaint Type 'MAVQ' was added to Field 21 list on Jan. 17, 2014
8. Field 48 was added on Apr. 24, 2014
9. Field 49 was added on Sept. 29, 2015
10. Flat file content changes May 17 - June 17, 2021
11. Information message updated on June 28, 2021
12. Fields 50-51 were added on April 30, 2026

May 17 - June 17, 2021 - Flat file content changes.
* Previously blank Y/N

In [23]:

# A3.5 - Extract complaints for target makes and model years

import pandas as pd
import zipfile

complaints_zip = (
    ROOT / "data" / "raw" / "Complaints" / "FLAT_CMPL.zip"
)

target_makes = {
    "AUDI", "BMW", "FORD", "HONDA", "HYUNDAI",
    "MERCEDES-BENZ", "NISSAN", "TOYOTA",
    "VOLKSWAGEN"
}

# Zero-based positions from the official 51-field dictionary
column_positions = [
    0, 1, 3, 4, 5, 6, 7, 8, 9, 10,
    11, 15, 16, 19, 45
]

column_names = [
    "complaint_id", "odi_number", "make",
    "model", "model_year", "crash",
    "incident_date", "fire",
    "number_of_injuries", "number_of_deaths",
    "components", "date_added", "filed_date",
    "summary", "product_type"
]

selected_chunks = []

with zipfile.ZipFile(complaints_zip, "r") as z:
    with z.open("FLAT_CMPL.txt") as f:

        reader = pd.read_csv(
            f,
            sep="\t",
            header=None,
            usecols=column_positions,
            dtype=str,
            encoding="latin-1",
            chunksize=100000
        )

        for chunk in reader:

            chunk.columns = column_names

            chunk["make"] = (
                chunk["make"]
                .str.strip()
                .str.upper()
            )

            chunk["model_year"] = pd.to_numeric(
                chunk["model_year"],
                errors="coerce"
            )

            # Keep target model years and vehicle complaints
            year_mask = chunk["model_year"].between(2015, 2023)
            vehicle_mask = chunk["product_type"] == "V"

            # Include alternative Mercedes naming for review
            make_mask = (
                chunk["make"].isin(target_makes) |
                chunk["make"].str.contains(
                    "MERCEDES", case=False, na=False
                )
            )

            selected = chunk.loc[
                year_mask & vehicle_mask & make_mask
            ].copy()

            if not selected.empty:
                selected_chunks.append(selected)

complaints_candidates = pd.concat(
    selected_chunks,
    ignore_index=True
)

print("Candidate complaint rows:", len(complaints_candidates))
print(
    "Model year range:",
    complaints_candidates["model_year"].min(),
    "-",
    complaints_candidates["model_year"].max()
)

display(
    complaints_candidates["make"]
    .value_counts()
)

display(complaints_candidates.head())


Candidate complaint rows: 248220
Model year range: 2015.0 - 2023.0


make
FORD                89194
HONDA               48295
HYUNDAI             29243
TOYOTA              23958
NISSAN              21643
VOLKSWAGEN          13444
MERCEDES-BENZ        8621
BMW                  7616
AUDI                 6009
MERCEDES BENZ         143
MERCEDES ROYALE        31
MERCEDES-MAYBACH       22
MERCEDES                1
Name: count, dtype: int64

,complaint_id,odi_number,make,model,model_year,crash,incident_date,fire,number_of_injuries,number_of_deaths,components,date_added,filed_date,summary,product_type
0,784286,10327950,MERCEDES BENZ,redundant S550E,2015.0,Y,20100225,N,0,0,ELECTRONIC STABILITY CONTROL (ESC),20100428,20100428,I WAS ON A SNOW COVERED ROAD IN MY NEIGHBORHOO...,V
1,784287,10327950,MERCEDES BENZ,redundant S550E,2015.0,Y,20100225,N,0,0,"SERVICE BRAKES, HYDRAULIC",20100428,20100428,I WAS ON A SNOW COVERED ROAD IN MY NEIGHBORHOO...,V
2,1078916,10605566,HONDA,FIT,2015.0,N,20140620,N,0,0,SEAT BELTS,20140624,20140624,THE SEAT BELT BUCKLES ON THE REAR CENTER AND R...,V
3,1079372,10605841,HONDA,FIT,2015.0,N,20140625,N,0,0,SEAT BELTS,20140625,20140625,THERE IS AN ISSUE WITH THE SEAT BELT BRACKET F...,V
4,1079385,10605842,HONDA,FIT,2015.0,N,20140611,N,0,0,SEAT BELTS,20140625,20140625,SEATBELT BUCKLES IN CENTER OF BACK SEATS ARE B...,V


In [24]:

# A3.6 - Standardize and filter target model families

complaints_work = complaints_candidates.copy()

# Standardize strings
complaints_work["make"] = (
    complaints_work["make"].astype("string")
    .str.strip().str.upper()
    .str.replace(r"\s+", " ", regex=True)
)

complaints_work["model"] = (
    complaints_work["model"].astype("string")
    .str.strip().str.upper()
    .str.replace(r"\s+", " ", regex=True)
)

# Normalize verified Mercedes make variants only
complaints_work["make"] = complaints_work["make"].replace({
    "MERCEDES BENZ": "MERCEDES-BENZ"
})

complaints_work["model_family"] = complaints_work["model"].copy()

# Apply BMW / Mercedes / Audi / Toyota model aliases
for make, mapping in model_aliases.items():
    mask = complaints_work["make"] == make

    complaints_work.loc[mask, "model_family"] = (
        complaints_work.loc[mask, "model"].replace(mapping)
    )

# Explicit target families
target_families = {
    ("AUDI", "Q5"),
    ("BMW", "3 SERIES"),
    ("FORD", "ESCAPE"),
    ("FORD", "F-150"),
    ("HONDA", "CIVIC"),
    ("HYUNDAI", "ELANTRA"),
    ("MERCEDES-BENZ", "C-CLASS"),
    ("NISSAN", "ALTIMA"),
    ("TOYOTA", "CAMRY"),
    ("VOLKSWAGEN", "TIGUAN")
}

target_index = pd.MultiIndex.from_tuples(
    target_families, names=["make", "model_family"]
)

complaints_final_candidates = complaints_work.loc[
    pd.MultiIndex.from_frame(
        complaints_work[["make", "model_family"]]
    ).isin(target_index)
].copy()

print("Target complaint rows:", len(complaints_final_candidates))

print("Unique model families:",
      complaints_final_candidates[
          ["make", "model_family"]
      ].drop_duplicates().shape[0])

print("Model year range:",
      complaints_final_candidates["model_year"].min(),
      "-",
      complaints_final_candidates["model_year"].max())

display(
    complaints_final_candidates
    .groupby(["make", "model_family"])
    .size()
    .reset_index(name="complaints")
    .sort_values("complaints", ascending=False)
)


Target complaint rows: 50465
Unique model families: 10
Model year range: 2015.0 - 2023.0


,make,model_family,complaints
3,FORD,F-150,15290
2,FORD,ESCAPE,12403
4,HONDA,CIVIC,6193
7,NISSAN,ALTIMA,3723
5,HYUNDAI,ELANTRA,3471
8,TOYOTA,CAMRY,3451
9,VOLKSWAGEN,TIGUAN,2166
6,MERCEDES-BENZ,C-CLASS,1754
1,BMW,3 SERIES,1030
0,AUDI,Q5,984


In [25]:

# A3.7 - Check potentially missed model names

patterns = {
    "AUDI": r"Q5",
    "BMW": r"^(3 SERIES|3-SERIES|3[0-9]{2}|M340|ACTIVEHYBRID 3)",
    "FORD": r"F.?150|ESCAPE",
    "HONDA": r"CIVIC",
    "HYUNDAI": r"ELANTRA",
    "MERCEDES-BENZ": r"^(C.CLASS|C\s?[0-9]{2,3}|AMG C\s?[0-9]{2})",
    "NISSAN": r"ALTIMA",
    "TOYOTA": r"CAMRY",
    "VOLKSWAGEN": r"TIGUAN"
}

for make, pattern in patterns.items():

    candidates = complaints_work.loc[
        (complaints_work["make"] == make) &
        complaints_work["model"].str.contains(
            pattern, regex=True, na=False
        )
    ]

    missed = candidates.loc[
        ~candidates.index.isin(
            complaints_final_candidates.index
        )
    ]

    print(f"\n===== {make} =====")
    print("Potentially missed rows:", len(missed))

    display(
        missed["model"]
        .value_counts()
        .head(20)
    )



===== AUDI =====
Potentially missed rows: 275


model
SQ5             141
Q5 E             81
Q5 SPORTBACK     53
Name: count, dtype: Int64


===== BMW =====
Potentially missed rows: 0


/var/folders/wz/6yh7tpb961s6040jz54nlnmh0000gn/T/ipykernel_44313/897085623.py:19: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  complaints_work["model"].str.contains(


Series([], Name: count, dtype: Int64)


===== FORD =====
Potentially missed rows: 2033


... (10 outputs left out)


===== TOYOTA =====
Potentially missed rows: 0


Series([], Name: count, dtype: Int64)


===== VOLKSWAGEN =====
Potentially missed rows: 12


model
TIGUAN LIMITED    12
Name: count, dtype: Int64

In [26]:

# A3.8 - Add verified family-level aliases

additional_aliases = {
    "AUDI": {
        "Q5 E": "Q5"
    },
    "FORD": {
        "F-150 HYBRID": "F-150",
        "ESCAPE HYBRID": "ESCAPE",
        "ESCAPE PHEV": "ESCAPE"
    },
    "HONDA": {
        "CIVIC SI": "CIVIC",
        "CIVIC TYPE R": "CIVIC",
        "CIVIC HATCHBACK": "CIVIC",
        "CIVIC HYBRID": "CIVIC"
    },
    "HYUNDAI": {
        "ELANTRA HYBRID": "ELANTRA",
        "ELANTRA GT": "ELANTRA",
        "ELANTRA N": "ELANTRA"
    },
    "MERCEDES-BENZ": {
        "C200": "C-CLASS",
        "C 300 4MATIC": "C-CLASS",
        "C220": "C-CLASS",
        "C 350 E": "C-CLASS",
        "AMG C63 S": "C-CLASS",
        "C205": "C-CLASS",
        "C 250 D": "C-CLASS",
        "C 350": "C-CLASS"
    },
    "VOLKSWAGEN": {
        "TIGUAN LIMITED": "TIGUAN"
    }
}

for make, mapping in additional_aliases.items():
    mask = complaints_work["make"] == make

    complaints_work.loc[mask, "model_family"] = (
        complaints_work.loc[mask, "model_family"]
        .replace(mapping)
    )

print("Additional model aliases applied.")


Additional model aliases applied.


In [27]:

# A3.9 - Refilter target model families

complaints_final_candidates_v2 = complaints_work.loc[
    pd.MultiIndex.from_frame(
        complaints_work[["make", "model_family"]]
    ).isin(target_index)
].copy()

print(
    "Previous candidate rows:",
    len(complaints_final_candidates)
)

print(
    "Updated candidate rows:",
    len(complaints_final_candidates_v2)
)

print(
    "Additional matched rows:",
    len(complaints_final_candidates_v2)
    - len(complaints_final_candidates)
)

print(
    "Unique model families:",
    complaints_final_candidates_v2[
        ["make", "model_family"]
    ].drop_duplicates().shape[0]
)

display(
    complaints_final_candidates_v2
    .groupby(["make", "model_family"])
    .size()
    .reset_index(name="complaints")
    .sort_values("complaints", ascending=False)
)


Previous candidate rows: 50465
Updated candidate rows: 52997
Additional matched rows: 2532
Unique model families: 10


,make,model_family,complaints
3,FORD,F-150,16145
2,FORD,ESCAPE,13224
4,HONDA,CIVIC,6456
5,HYUNDAI,ELANTRA,3888
7,NISSAN,ALTIMA,3723
8,TOYOTA,CAMRY,3451
9,VOLKSWAGEN,TIGUAN,2178
6,MERCEDES-BENZ,C-CLASS,1837
0,AUDI,Q5,1065
1,BMW,3 SERIES,1030


In [28]:

# A3.10 - Compare final candidates with midterm data

# Original midterm complaints
midterm_ids = set(
    df["complaint_id"]
    .dropna()
    .astype(str)
    .str.strip()
)

# New expanded complaints
final_ids = set(
    complaints_final_candidates_v2["complaint_id"]
    .dropna()
    .astype(str)
    .str.strip()
)

# Compare IDs
overlap_ids = midterm_ids & final_ids
missing_ids = midterm_ids - final_ids

print("Midterm unique complaint IDs:", len(midterm_ids))
print("Expanded unique complaint IDs:", len(final_ids))
print("Midterm IDs found in expanded data:", len(overlap_ids))
print("Midterm IDs not found:", len(missing_ids))

if len(midterm_ids) > 0:
    print(
        "Midterm coverage:",
        round(len(overlap_ids) / len(midterm_ids) * 100, 2),
        "%"
    )

# Check possible duplicate IDs in expanded data
print(
    "Duplicate complaint IDs in expanded data:",
    complaints_final_candidates_v2["complaint_id"]
    .duplicated().sum()
)


Midterm unique complaint IDs: 9461
Expanded unique complaint IDs: 52997
Midterm IDs found in expanded data: 0
Midterm IDs not found: 9461
Midterm coverage: 0.0 %
Duplicate complaint IDs in expanded data: 0


In [29]:

# A3.11 - Diagnose complaint ID mismatch

print("===== MIDTERM =====")
display(
    df[
        ["complaint_id", "make", "model", "model_year"]
    ].head(10)
)

print("\n===== EXPANDED =====")
display(
    complaints_final_candidates_v2[
        ["complaint_id", "odi_number", "make",
         "model", "model_year"]
    ].head(10)
)

# Compare midterm IDs against both NHTSA identifiers
midterm_ids = set(
    df["complaint_id"].dropna().astype(str).str.strip()
)

cmpl_ids = set(
    complaints_final_candidates_v2["complaint_id"]
    .dropna().astype(str).str.strip()
)

odi_ids = set(
    complaints_final_candidates_v2["odi_number"]
    .dropna().astype(str).str.strip()
)

print("\nOverlap with CMPLID:", len(midterm_ids & cmpl_ids))
print("Overlap with ODINO:", len(midterm_ids & odi_ids))


===== MIDTERM =====


,complaint_id,make,model,model_year
0,11770133,TOYOTA,CAMRY,2018
1,11769969,TOYOTA,CAMRY,2018
2,11769124,TOYOTA,CAMRY,2018
3,11768868,TOYOTA,CAMRY,2018
4,11768335,TOYOTA,CAMRY,2018
5,11767971,TOYOTA,CAMRY,2018
6,11766991,TOYOTA,CAMRY,2018
7,11766206,TOYOTA,CAMRY,2018
8,11766000,TOYOTA,CAMRY,2018
9,11765476,TOYOTA,CAMRY,2018



===== EXPANDED =====


,complaint_id,odi_number,make,model,model_year
18,1114904,10640005,NISSAN,ALTIMA,2015.0
43,1129377,10654095,HYUNDAI,ELANTRA,2015.0
44,1129814,10654431,BMW,328XI,2015.0
50,1134567,10660936,AUDI,Q5,2015.0
62,1137299,10662944,TOYOTA,CAMRY,2015.0
77,1139406,10664470,MERCEDES-BENZ,C300,2015.0
78,1139407,10664470,MERCEDES-BENZ,C300,2015.0
79,1140300,10667098,HYUNDAI,ELANTRA,2015.0
81,1141096,10667633,FORD,ESCAPE,2015.0
92,1143202,10669168,NISSAN,ALTIMA,2015.0



Overlap with CMPLID: 0
Overlap with ODINO: 9431


In [30]:

# A3.12 - Inspect missing midterm complaint IDs

midterm_ids = set(
    df["complaint_id"]
    .dropna()
    .astype(str)
    .str.strip()
)

expanded_odi_ids = set(
    complaints_final_candidates_v2["odi_number"]
    .dropna()
    .astype(str)
    .str.strip()
)

missing_ids = midterm_ids - expanded_odi_ids

print("Missing midterm IDs:", len(missing_ids))

missing_midterm = df.loc[
    df["complaint_id"]
    .astype(str)
    .str.strip()
    .isin(missing_ids)
].copy()

display(
    missing_midterm[
        [
            "complaint_id",
            "make",
            "model",
            "model_year",
            "filed_date"
        ]
    ].sort_values(["make", "model", "model_year"])
)

print("\nMissing IDs by vehicle:")
display(
    missing_midterm
    .groupby(["make", "model"])
    .size()
    .reset_index(name="missing_count")
)


Missing midterm IDs: 30


,complaint_id,make,model,model_year,filed_date
2686,11748293,FORD,ESCAPE,2018,2026-07-04
3897,11549321,FORD,ESCAPE,2018,2023-10-11
4380,11161018,FORD,ESCAPE,2018,2018-12-12
6065,11547894,FORD,ESCAPE,2020,2023-10-03
6299,11522044,FORD,ESCAPE,2020,2023-05-15
7766,11713820,NISSAN,ALTIMA,2019,2026-01-28
8028,11628888,NISSAN,ALTIMA,2020,2024-12-04
8030,11625048,NISSAN,ALTIMA,2020,2024-11-13
8034,11612407,NISSAN,ALTIMA,2020,2024-09-03
8041,11598580,NISSAN,ALTIMA,2020,2024-07-03



Missing IDs by vehicle:


,make,model,missing_count
0,FORD,ESCAPE,5
1,NISSAN,ALTIMA,22
2,TOYOTA,CAMRY,1
3,VOLKSWAGEN,TIGUAN,2


In [31]:

# A3.13 - Locate missing complaints before model matching

missing_raw = complaints_candidates.loc[
    complaints_candidates["odi_number"]
    .astype(str)
    .str.strip()
    .isin(missing_ids)
].copy()

print("Missing midterm IDs:", len(missing_ids))

print(
    "IDs found before model matching:",
    missing_raw["odi_number"].nunique()
)

print(
    "IDs still not found:",
    len(
        missing_ids -
        set(missing_raw["odi_number"].astype(str).str.strip())
    )
)

display(
    missing_raw[
        [
            "odi_number",
            "make",
            "model",
            "model_year",
            "filed_date",
            "product_type"
        ]
    ]
    .drop_duplicates()
    .sort_values(["make", "model", "model_year"])
)


Missing midterm IDs: 30
IDs found before model matching: 21
IDs still not found: 9


,odi_number,make,model,model_year,filed_date,product_type
70574,11421953,NISSAN,redundant ALTIMA,2020.0,20210623,V
72692,11428499,NISSAN,redundant ALTIMA,2020.0,20210810,V
81100,11452018,NISSAN,redundant ALTIMA,2020.0,20220215,V
82697,11455561,NISSAN,redundant ALTIMA,2020.0,20220307,V
100272,11493869,NISSAN,redundant ALTIMA,2020.0,20221116,V
106932,11507402,NISSAN,redundant ALTIMA,2020.0,20230215,V
112579,11518324,NISSAN,redundant ALTIMA,2020.0,20230421,V
120021,11532314,NISSAN,redundant ALTIMA,2020.0,20230714,V
124066,11539744,NISSAN,redundant ALTIMA,2020.0,20230821,V
124580,11540567,NISSAN,redundant ALTIMA,2020.0,20230824,V


In [32]:

# A3.14 - Add redundant ALTIMA as a candidate alias

mask = (
    (complaints_work["make"] == "NISSAN") &
    (complaints_work["model"] == "REDUNDANT ALTIMA")
)

print("Records with REDUNDANT ALTIMA:", mask.sum())

# Preserve the original model name
complaints_work.loc[mask, "model_family"] = "ALTIMA"

# Refilter the target families
complaints_final_candidates_v3 = complaints_work.loc[
    pd.MultiIndex.from_frame(
        complaints_work[["make", "model_family"]]
    ).isin(target_index)
].copy()

print("Previous candidate rows:", len(complaints_final_candidates_v2))
print("Updated candidate rows:", len(complaints_final_candidates_v3))
print(
    "Newly included rows:",
    len(complaints_final_candidates_v3) -
    len(complaints_final_candidates_v2)
)


Records with REDUNDANT ALTIMA: 26
Previous candidate rows: 52997
Updated candidate rows: 53023
Newly included rows: 26


In [33]:

# A3.15 - Recheck midterm complaint coverage

midterm_ids = set(
    df["complaint_id"].dropna().astype(str).str.strip()
)

expanded_ids = set(
    complaints_final_candidates_v3["odi_number"]
    .dropna().astype(str).str.strip()
)

matched_ids = midterm_ids & expanded_ids
remaining_ids = midterm_ids - expanded_ids

print("Midterm unique IDs:", len(midterm_ids))
print("Matched IDs:", len(matched_ids))
print("Remaining unmatched IDs:", len(remaining_ids))

print(
    "Coverage:",
    round(100 * len(matched_ids) / len(midterm_ids), 2),
    "%"
)

print("Remaining IDs:", sorted(remaining_ids))


Midterm unique IDs: 9461
Matched IDs: 9452
Remaining unmatched IDs: 9
Coverage: 99.9 %
Remaining IDs: ['11111398', '11161018', '11522044', '11547894', '11549321', '11713820', '11746820', '11748293', '11757871']


In [34]:

# A3.16 - Search the full raw dataset for 9 missing ODINO values

import zipfile
import pandas as pd

remaining_ids = midterm_ids - expanded_ids

found_chunks = []

with zipfile.ZipFile(complaints_zip, "r") as z:
    with z.open("FLAT_CMPL.txt") as f:
        reader = pd.read_csv(
            f,
            sep="\t",
            header=None,
            usecols=[0, 1, 3, 4, 5, 16, 45],
            dtype=str,
            encoding="latin-1",
            chunksize=100000
        )

        for chunk in reader:
            chunk.columns = [
                "cmplid", "odi_number", "make",
                "model", "model_year",
                "filed_date", "product_type"
            ]

            mask = (
                chunk["odi_number"]
                .str.strip()
                .isin(remaining_ids)
            )

            if mask.any():
                found_chunks.append(chunk.loc[mask].copy())

if found_chunks:
    missing_raw_full = pd.concat(
        found_chunks,
        ignore_index=True
    )
else:
    missing_raw_full = pd.DataFrame(
        columns=[
            "cmplid", "odi_number", "make",
            "model", "model_year",
            "filed_date", "product_type"
        ]
    )

found_ids = set(
    missing_raw_full["odi_number"].dropna().str.strip()
)

print("Missing IDs searched:", len(remaining_ids))
print("Found in full raw dataset:", len(found_ids))
print("Not found anywhere:", len(remaining_ids - found_ids))

display(missing_raw_full)


Missing IDs searched: 9
Found in full raw dataset: 0
Not found anywhere: 9


,cmplid,odi_number,make,model,model_year,filed_date,product_type


In [35]:

# A3.17 - Check duplicate complaints and date quality

complaints_quality = complaints_final_candidates_v3.copy()

# Convert date fields
complaints_quality["filed_date"] = pd.to_datetime(
    complaints_quality["filed_date"],
    format="%Y%m%d",
    errors="coerce"
)

complaints_quality["incident_date"] = pd.to_datetime(
    complaints_quality["incident_date"],
    format="%Y%m%d",
    errors="coerce"
)

# Check duplicates
print("Total rows:", len(complaints_quality))

print(
    "Unique CMPLID:",
    complaints_quality["complaint_id"].nunique()
)

print(
    "Unique ODINO:",
    complaints_quality["odi_number"].nunique()
)

print(
    "Duplicated CMPLID rows:",
    complaints_quality["complaint_id"].duplicated().sum()
)

print(
    "Duplicated ODINO rows:",
    complaints_quality["odi_number"].duplicated().sum()
)

# Check dates
print("\n===== DATE QUALITY =====")

print(
    "Missing filed dates:",
    complaints_quality["filed_date"].isna().sum()
)

print(
    "Missing incident dates:",
    complaints_quality["incident_date"].isna().sum()
)

print(
    "Earliest filed date:",
    complaints_quality["filed_date"].min()
)

print(
    "Latest filed date:",
    complaints_quality["filed_date"].max()
)

print(
    "Incident dates after filed dates:",
    (
        complaints_quality["incident_date"] >
        complaints_quality["filed_date"]
    ).sum()
)


Total rows: 53023
Unique CMPLID: 53023
Unique ODINO: 39905
Duplicated CMPLID rows: 0
Duplicated ODINO rows: 13118

===== DATE QUALITY =====
Missing filed dates: 0
Missing incident dates: 0
Earliest filed date: 2014-09-29 00:00:00
Latest filed date: 2026-11-16 00:00:00
Incident dates after filed dates: 0


In [36]:

# A3.18 - Inspect repeated ODINO records

# Number of rows per ODINO
odi_counts = (
    complaints_quality
    .groupby("odi_number")
    .size()
    .sort_values(ascending=False)
)

print("Unique ODINO:", len(odi_counts))
print("ODINO appearing more than once:", (odi_counts > 1).sum())
print("Maximum rows for one ODINO:", odi_counts.max())

print("\n===== RECORDS PER ODINO =====")
display(odi_counts.value_counts().sort_index().head(15))

# Check whether one ODINO belongs to multiple vehicle groups
group_counts = (
    complaints_quality
    .groupby("odi_number")
    .apply(
        lambda g: g[
            ["make", "model_family", "model_year"]
        ].drop_duplicates().shape[0],
        include_groups=False
    )
)

print(
    "\nODINO linked to multiple vehicle groups:",
    (group_counts > 1).sum()
)

# Show one example of a repeated ODINO
example_odi = odi_counts.index[0]

print("\n===== EXAMPLE REPEATED ODINO =====")
print("ODINO:", example_odi)

display(
    complaints_quality.loc[
        complaints_quality["odi_number"] == example_odi,
        [
            "complaint_id", "odi_number",
            "make", "model_family", "model_year",
            "components", "crash", "fire",
            "number_of_injuries", "number_of_deaths"
        ]
    ]
)


Unique ODINO: 39905
ODINO appearing more than once: 9729
Maximum rows for one ODINO: 6

===== RECORDS PER ODINO =====


1    30176
2     6363
3     3351
4        8
5        6
6        1
Name: count, dtype: int64


ODINO linked to multiple vehicle groups: 0

===== EXAMPLE REPEATED ODINO =====
ODINO: 10807026


,complaint_id,odi_number,make,model_family,model_year,components,crash,fire,number_of_injuries,number_of_deaths
2180,1245731,10807026,FORD,F-150,2015.0,AIR BAGS,N,N,0,0
2181,1245732,10807026,FORD,F-150,2015.0,ELECTRICAL SYSTEM,N,N,0,0
2182,1245733,10807026,FORD,F-150,2015.0,TIRES,N,N,0,0
2183,1245734,10807026,FORD,F-150,2015.0,EQUIPMENT:APPLIANCE:AIR CONDITIONER,N,N,0,0
2184,1245735,10807026,FORD,F-150,2015.0,POWER TRAIN,N,N,0,0
2185,1245736,10807026,FORD,F-150,2015.0,STRUCTURE,N,N,0,0


In [37]:

# A3.19 - Aggregate multiple component rows by ODINO

import pandas as pd

complaints_agg_work = complaints_quality.copy()

# Convert numeric severity fields
for col in ["number_of_injuries", "number_of_deaths"]:
    complaints_agg_work[col] = pd.to_numeric(
        complaints_agg_work[col],
        errors="coerce"
    )

# Normalize crash and fire indicators
for col in ["crash", "fire"]:
    complaints_agg_work[col] = (
        complaints_agg_work[col]
        .astype("string")
        .str.strip()
        .str.upper()
    )

def combine_components(series):
    values = (
        series.dropna()
        .astype(str)
        .str.strip()
    )

    values = sorted(set(
        value for value in values if value
    ))

    return " | ".join(values)

def any_yes(series):
    return "Y" if series.eq("Y").any() else "N"

complaints_aggregated = (
    complaints_agg_work
    .sort_values(["odi_number", "complaint_id"])
    .groupby("odi_number", as_index=False)
    .agg(
        cmplid_first=("complaint_id", "first"),
        make=("make", "first"),
        model=("model", "first"),
        model_family=("model_family", "first"),
        model_year=("model_year", "first"),
        filed_date=("filed_date", "min"),
        incident_date=("incident_date", "min"),
        components=("components", combine_components),
        crash=("crash", any_yes),
        fire=("fire", any_yes),
        number_of_injuries=("number_of_injuries", "max"),
        number_of_deaths=("number_of_deaths", "max"),
        summary=("summary", "first")
    )
)

# Use ODINO as the complaint identifier
complaints_aggregated = complaints_aggregated.rename(
    columns={"odi_number": "complaint_id"}
)

print("Before aggregation:", len(complaints_agg_work))
print("After aggregation:", len(complaints_aggregated))

print(
    "Unique complaint IDs:",
    complaints_aggregated["complaint_id"].nunique()
)

display(complaints_aggregated.head())


Before aggregation: 53023
After aggregation: 39905
Unique complaint IDs: 39905


,complaint_id,cmplid_first,make,model,model_family,model_year,filed_date,incident_date,components,crash,fire,number_of_injuries,number_of_deaths,summary
0,10640005,1114904,NISSAN,ALTIMA,ALTIMA,2015.0,2014-09-29,2014-09-29,VISIBILITY/WIPER,N,N,0,0,THE WINDSHIELD CRACKED FOR NO GOOD REASON. TO...
1,10654095,1129377,HYUNDAI,ELANTRA,ELANTRA,2015.0,2014-11-11,2014-09-17,STEERING,N,N,0,0,"THE STEERING COLUMN HAVE UNUSUAL OVERHEAT, THE..."
2,10654431,1129814,BMW,328XI,3 SERIES,2015.0,2014-11-12,2014-11-07,SERVICE BRAKES,N,N,0,0,THIS IS A BRAND NEW VEHICLE LEASE BMW 2015 328...
3,10660936,1134567,AUDI,Q5,Q5,2015.0,2014-11-25,2014-11-23,VISIBILITY:SUN/MOON ROOF ASSEMBLY,N,N,0,0,APPROXIMATELY 10:45 AM MY HUSBAND (THE DRIVER)...
4,10662944,1137299,TOYOTA,CAMRY,CAMRY,2015.0,2014-12-08,2014-11-12,VISIBILITY/WIPER,N,N,0,0,WHEN DRIVING ON A SUNNY DAY---OR AT NIGHT ON A...


In [38]:

# A3.20 - Final complaint-level data quality check

print("===== BASIC CHECK =====")

print("Total complaints:", len(complaints_aggregated))

print(
    "Duplicate complaint IDs:",
    complaints_aggregated["complaint_id"].duplicated().sum()
)

print(
    "Unique model families:",
    complaints_aggregated[
        ["make", "model_family"]
    ].drop_duplicates().shape[0]
)

print(
    "Unique vehicle groups:",
    complaints_aggregated[
        ["make", "model_family", "model_year"]
    ].drop_duplicates().shape[0]
)

print("\n===== MISSING VALUES =====")

cols_to_check = [
    "make", "model_family", "model_year",
    "filed_date", "incident_date",
    "components", "crash", "fire",
    "number_of_injuries", "number_of_deaths"
]

display(
    complaints_aggregated[cols_to_check]
    .isna()
    .sum()
    .to_frame("missing_count")
)

print("\n===== DATE CHECK =====")

print(
    "Filed date range:",
    complaints_aggregated["filed_date"].min(),
    "to",
    complaints_aggregated["filed_date"].max()
)

print(
    "Incident date after filed date:",
    (
        complaints_aggregated["incident_date"] >
        complaints_aggregated["filed_date"]
    ).sum()
)

print("\n===== SEVERITY CHECK =====")

for col in ["number_of_injuries", "number_of_deaths"]:
    print(
        col,
        "- min:", complaints_aggregated[col].min(),
        "max:", complaints_aggregated[col].max()
    )


===== BASIC CHECK =====
Total complaints: 39905
Duplicate complaint IDs: 0
Unique model families: 10
Unique vehicle groups: 90

===== MISSING VALUES =====


,missing_count
make,0
model_family,0
model_year,0
filed_date,0
incident_date,0
components,0
crash,0
fire,0
number_of_injuries,0
number_of_deaths,0



===== DATE CHECK =====
Filed date range: 2014-09-29 00:00:00 to 2026-11-16 00:00:00
Incident date after filed date: 0

===== SEVERITY CHECK =====
number_of_injuries - min: 0 max: 14
number_of_deaths - min: 0 max: 2


In [39]:

# A3.21 - Inspect midterm reporting delay

print("Midterm columns:")
print(df.columns.tolist())

print("\nReporting delay sample:")
display(
    df[
        [
            "complaint_id",
            "filed_date",
            "incident_date",
            "reporting_delay",
            "invalid_incident_date"
        ]
    ].head(10)
)

print("\nReporting delay statistics:")
display(df["reporting_delay"].describe())

print("\nInvalid incident date values:")
display(
    df["invalid_incident_date"]
    .value_counts(dropna=False)
)


Midterm columns:
['complaint_id', 'make', 'model', 'model_year', 'product_model', 'filed_date', 'incident_date', 'components', 'crash', 'fire', 'number_of_injuries', 'number_of_deaths', 'summary', 'reporting_delay', 'invalid_incident_date']

Reporting delay sample:


,complaint_id,filed_date,incident_date,reporting_delay,invalid_incident_date
0,11770133,2026-10-09,2026-09-29,10.0,False
1,11769969,2026-10-08,2026-10-06,2.0,False
2,11769124,2026-10-05,2026-10-03,2.0,False
3,11768868,2026-10-04,2026-10-02,2.0,False
4,11768335,2026-10-01,2026-09-29,2.0,False
5,11767971,2026-09-30,2026-09-28,2.0,False
6,11766991,2026-09-25,2026-09-06,19.0,False
7,11766206,2026-09-22,2026-09-22,0.0,False
8,11766000,2026-09-21,2026-03-19,186.0,False
9,11765476,2026-09-19,2026-07-24,57.0,False



Reporting delay statistics:


count    9434.000000
mean       89.878100
std       229.309075
min         0.000000
25%         2.000000
50%        12.000000
75%        60.750000
max      2922.000000
Name: reporting_delay, dtype: float64


Invalid incident date values:


invalid_incident_date
False    9434
True       27
Name: count, dtype: int64

In [40]:

# A3.22 - Find original midterm date cleaning logic

import nbformat

midterm_notebook = (
    ROOT / "src" / "preprocessing"
    / "00_shared_preprocessing.ipynb"
)

nb = nbformat.read(midterm_notebook, as_version=4)

keywords = [
    "reporting_delay",
    "invalid_incident_date",
    "incident_date"
]

for i, cell in enumerate(nb.cells):
    if cell.cell_type != "code":
        continue

    source = cell.source

    if any(keyword in source for keyword in keywords):
        print(f"\n{'=' * 20} CELL {i} {'=' * 20}")
        print(source[:5000])



==================== CELL 40 ====================
df_clean["filed_date"] = pd.to_datetime(
    df_clean["dateComplaintFiled"],
    format="%m/%d/%Y",
    errors="coerce"
)

df_clean["incident_date"] = pd.to_datetime(
    df_clean["dateOfIncident"],
    format="%m/%d/%Y",
    errors="coerce"
)

==================== CELL 42 ====================
print("Missing filed dates:", df_clean["filed_date"].isna().sum())
print("Missing incident dates:", df_clean["incident_date"].isna().sum())

print("Implausible incident dates:",
      (df_clean["incident_date"].dt.year <
       df_clean["model_year"] - 2).sum())

print("Incident after filing:",
      (df_clean["incident_date"] > df_clean["filed_date"]).sum())

==================== CELL 43 ====================
df_clean["reporting_delay"] = (
    df_clean["filed_date"] - df_clean["incident_date"]
).dt.days

df_clean["reporting_delay"].describe()

==================== CELL 44 ====================
df_clean.loc[
    df_clean["incident_date"].dt.year <

In [41]:

# A3.23 - Apply midterm date cleaning rules

complaints_final = complaints_aggregated.copy()

# Ensure date columns use datetime
complaints_final["filed_date"] = pd.to_datetime(
    complaints_final["filed_date"],
    errors="coerce"
)

complaints_final["incident_date"] = pd.to_datetime(
    complaints_final["incident_date"],
    errors="coerce"
)

# Calculate reporting delay in days
complaints_final["reporting_delay"] = (
    complaints_final["filed_date"] -
    complaints_final["incident_date"]
).dt.days

# Flag implausibly early incident dates
complaints_final["invalid_incident_date"] = (
    complaints_final["incident_date"].dt.year <
    complaints_final["model_year"] - 2
)

# Apply the same cleaning rule as midterm
complaints_final["reporting_delay"] = (
    complaints_final["reporting_delay"].where(
        ~complaints_final["invalid_incident_date"]
    )
)

print("Total complaints:", len(complaints_final))

print(
    "Invalid incident dates:",
    complaints_final["invalid_incident_date"].sum()
)

print(
    "Valid reporting delays:",
    complaints_final["reporting_delay"].notna().sum()
)

print("\nReporting delay statistics:")
display(complaints_final["reporting_delay"].describe())


Total complaints: 39905
Invalid incident dates: 87
Valid reporting delays: 39818

Reporting delay statistics:


count    39818.000000
mean       106.834748
std        274.288202
min          0.000000
25%          1.000000
50%         13.000000
75%         72.000000
max       4448.000000
Name: reporting_delay, dtype: float64

In [42]:

# A3.24 - Save expanded complaints dataset

from pathlib import Path

output_dir = ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "complaints_final.csv"

# Select and organize final columns
final_columns = [
    "complaint_id",
    "cmplid_first",
    "make",
    "model",
    "model_family",
    "model_year",
    "filed_date",
    "incident_date",
    "components",
    "crash",
    "fire",
    "number_of_injuries",
    "number_of_deaths",
    "summary",
    "reporting_delay",
    "invalid_incident_date"
]

complaints_final = complaints_final[final_columns].copy()

# Standardize model year
complaints_final["model_year"] = (
    complaints_final["model_year"].astype(int)
)

# Verify before saving
assert complaints_final["complaint_id"].is_unique
assert complaints_final["filed_date"].notna().all()
assert complaints_final["model_year"].between(2015, 2023).all()

# Save without overwriting midterm data
complaints_final.to_csv(output_file, index=False)

print("Saved to:", output_file)
print("Final shape:", complaints_final.shape)
print("Unique complaints:", complaints_final["complaint_id"].nunique())


Saved to: /Users/xinyisong/Desktop/DATA1030-Midterm-Xinfinity/data/processed/complaints_final.csv
Final shape: (39905, 16)
Unique complaints: 39905


In [43]:

# A4.1 - Inspect investigation action types

inv_target_review = inv_matched_final.copy()

# Extract investigation action type
inv_target_review["action_type"] = (
    inv_target_review["action_number"]
    .astype("string")
    .str[:2]
)

# One action number should only be counted once
unique_actions = (
    inv_target_review[
        ["action_number", "action_type", "open_date", "subject"]
    ]
    .drop_duplicates(subset=["action_number"])
    .sort_values("open_date")
)

print("Total unique action numbers:", len(unique_actions))

print("\n===== ACTION TYPE DISTRIBUTION =====")
display(
    unique_actions["action_type"]
    .value_counts()
    .rename_axis("action_type")
    .reset_index(name="count")
)

print("\n===== OPEN YEAR DISTRIBUTION =====")
display(
    unique_actions["open_date"]
    .dt.year
    .value_counts()
    .sort_index()
    .rename_axis("open_year")
    .reset_index(name="count")
)

print("\n===== ALL MATCHED ACTIONS =====")
display(
    unique_actions[
        ["action_number", "action_type",
         "open_date", "subject"]
    ]
)


Total unique action numbers: 33

===== ACTION TYPE DISTRIBUTION =====


,action_type,count
0,PE,11
1,EA,9
2,RQ,5
3,AQ,4
4,DP,4



===== OPEN YEAR DISTRIBUTION =====


,open_year,count
0,2016,2
1,2017,1
2,2018,4
3,2019,3
4,2021,4
5,2022,5
6,2023,9
7,2024,2
8,2025,1
9,2026,2



===== ALL MATCHED ACTIONS =====


,action_number,action_type,open_date,subject
169,PE16001,PE,2016-01-16,Door Latch Failure
214,RQ16001,RQ,2016-09-28,Brake Master Cylinder Failure
0,AQ17001,AQ,2017-01-04,Champion Ford Edinboro / Steering Shaft
171,PE18001,PE,2018-01-10,Seat Belt Webbing Failure
40,EA18005,EA,2018-07-03,Seat Belt Webbing Failure
172,PE18008,PE,2018-08-03,Seatbelt Pretensioner Fire
176,PE18013,PE,2018-12-01,Rear Suspension Control Arm Failure
43,EA19001,EA,2019-04-19,Electrical overstress
44,EA19002,EA,2019-07-29,Rear Suspension Control Arm Failure
222,RQ19005,RQ,2019-12-19,Door Latch Failures


In [44]:

# A4.2 - Build broad NHTSA action event table

broad_types = ["PE", "EA", "RQ", "AQ", "DP"]

broad_events = inv_matched_final.copy()

broad_events["action_type"] = (
    broad_events["action_number"]
    .astype("string")
    .str[:2]
)

broad_events = broad_events.loc[
    broad_events["action_type"].isin(broad_types)
    & broad_events["open_date"].notna()
].copy()

event_keys = [
    "make", "model_family", "model_year",
    "action_number"
]

broad_events = (
    broad_events
    .sort_values("open_date")
    .drop_duplicates(subset=event_keys)
    [
        event_keys +
        ["action_type", "open_date", "subject"]
    ]
    .reset_index(drop=True)
)

print("Action-vehicle events:", len(broad_events))
print("Unique action numbers:",
      broad_events["action_number"].nunique())

print("Unique vehicle groups with actions:",
      broad_events[
          ["make", "model_family", "model_year"]
      ].drop_duplicates().shape[0])

display(
    broad_events["action_type"]
    .value_counts()
)

display(broad_events.head(10))


Action-vehicle events: 104
Unique action numbers: 33
Unique vehicle groups with actions: 54


action_type
EA    31
PE    30
DP    27
RQ     8
AQ     8
Name: count, dtype: Int64

,make,model_family,model_year,action_number,action_type,open_date,subject
0,FORD,ESCAPE,2015.0,PE16001,PE,2016-01-16,Door Latch Failure
1,FORD,F-150,2015.0,RQ16001,RQ,2016-09-28,Brake Master Cylinder Failure
2,FORD,F-150,2016.0,RQ16001,RQ,2016-09-28,Brake Master Cylinder Failure
3,FORD,F-150,2015.0,AQ17001,AQ,2017-01-04,Champion Ford Edinboro / Steering Shaft
4,VOLKSWAGEN,TIGUAN,2018.0,PE18001,PE,2018-01-10,Seat Belt Webbing Failure
5,VOLKSWAGEN,TIGUAN,2018.0,EA18005,EA,2018-07-03,Seat Belt Webbing Failure
6,FORD,F-150,2015.0,PE18008,PE,2018-08-03,Seatbelt Pretensioner Fire
7,FORD,F-150,2016.0,PE18008,PE,2018-08-03,Seatbelt Pretensioner Fire
8,FORD,F-150,2018.0,PE18008,PE,2018-08-03,Seatbelt Pretensioner Fire
9,FORD,F-150,2017.0,PE18008,PE,2018-08-03,Seatbelt Pretensioner Fire


In [45]:

# A4.3 - Create quarterly snapshot grid

import pandas as pd

# All 90 target vehicle model-year groups
vehicle_groups = (
    complaints_final[
        ["make", "model_family", "model_year"]
    ]
    .drop_duplicates()
    .copy()
)

vehicle_groups["model_year"] = (
    vehicle_groups["model_year"].astype(int)
)

# Quarterly prediction dates
snapshot_dates = pd.DataFrame({
    "snapshot_date": pd.date_range(
        start="2016-01-01",
        end="2025-10-01",
        freq="QS"
    )
})

# Create all combinations
snapshot_grid = vehicle_groups.merge(
    snapshot_dates,
    how="cross"
)

# Do not create prediction rows before the vehicle model year
snapshot_grid = snapshot_grid.loc[
    snapshot_grid["snapshot_date"].dt.year
    >= snapshot_grid["model_year"]
].copy()

snapshot_grid = (
    snapshot_grid
    .sort_values(
        ["make", "model_family",
         "model_year", "snapshot_date"]
    )
    .reset_index(drop=True)
)

print("Unique vehicle groups:", len(vehicle_groups))
print("Quarterly dates:", len(snapshot_dates))
print("Total prediction samples:", len(snapshot_grid))

print(
    "Snapshot range:",
    snapshot_grid["snapshot_date"].min(),
    "to",
    snapshot_grid["snapshot_date"].max()
)

display(snapshot_grid.head(10))


Unique vehicle groups: 90
Quarterly dates: 40
Total prediction samples: 2480
Snapshot range: 2016-01-01 00:00:00 to 2025-10-01 00:00:00


,make,model_family,model_year,snapshot_date
0,AUDI,Q5,2015,2016-01-01
1,AUDI,Q5,2015,2016-04-01
2,AUDI,Q5,2015,2016-07-01
3,AUDI,Q5,2015,2016-10-01
4,AUDI,Q5,2015,2017-01-01
5,AUDI,Q5,2015,2017-04-01
6,AUDI,Q5,2015,2017-07-01
7,AUDI,Q5,2015,2017-10-01
8,AUDI,Q5,2015,2018-01-01
9,AUDI,Q5,2015,2018-04-01


In [46]:

# A4.4 - Create future 12-month investigation labels

import pandas as pd

vehicle_keys = ["make", "model_family", "model_year"]

# Start from the snapshot grid
labeled_grid = snapshot_grid.copy()

labeled_grid["window_end"] = (
    labeled_grid["snapshot_date"] +
    pd.DateOffset(months=12)
)

# Ensure event dates use datetime
events_for_labels = broad_events.copy()
events_for_labels["open_date"] = pd.to_datetime(
    events_for_labels["open_date"]
)

# Match actions to the same vehicle group
matched_windows = labeled_grid.merge(
    events_for_labels[
        vehicle_keys + ["action_number", "open_date"]
    ],
    on=vehicle_keys,
    how="left"
)

# Identify actions opened within the prediction window
matched_windows["event_in_window"] = (
    (matched_windows["open_date"] >=
     matched_windows["snapshot_date"]) &
    (matched_windows["open_date"] <
     matched_windows["window_end"])
)

# One positive event is enough for y = 1
target_labels = (
    matched_windows
    .groupby(vehicle_keys + ["snapshot_date"], as_index=False)
    .agg(
        future_investigation=(
            "event_in_window", "max"
        )
    )
)

target_labels["future_investigation"] = (
    target_labels["future_investigation"].astype(int)
)

print("Total labeled samples:", len(target_labels))

print("\n===== CLASS DISTRIBUTION =====")
display(
    target_labels["future_investigation"]
    .value_counts()
    .sort_index()
)

print("\n===== POSITIVE RATE =====")
print(
    round(
        target_labels["future_investigation"].mean() * 100,
        2
    ),
    "%"
)

display(target_labels.head(10))


Total labeled samples: 2480

===== CLASS DISTRIBUTION =====


future_investigation
0    2123
1     357
Name: count, dtype: int64


===== POSITIVE RATE =====
14.4 %


,make,model_family,model_year,snapshot_date,future_investigation
0,AUDI,Q5,2015,2016-01-01,0
1,AUDI,Q5,2015,2016-04-01,0
2,AUDI,Q5,2015,2016-07-01,0
3,AUDI,Q5,2015,2016-10-01,0
4,AUDI,Q5,2015,2017-01-01,0
5,AUDI,Q5,2015,2017-04-01,0
6,AUDI,Q5,2015,2017-07-01,0
7,AUDI,Q5,2015,2017-10-01,0
8,AUDI,Q5,2015,2018-01-01,0
9,AUDI,Q5,2015,2018-04-01,0


In [47]:

# A5.1 - Count complaints in the previous 12 months

import pandas as pd

vehicle_keys = ["make", "model_family", "model_year"]

# Start from labeled snapshots
feature_grid = target_labels.copy()

# Ensure dates are datetime
complaints_features = complaints_final.copy()

complaints_features["filed_date"] = pd.to_datetime(
    complaints_features["filed_date"]
)

# Match complaints to their vehicle group
complaint_windows = feature_grid.merge(
    complaints_features[
        vehicle_keys + ["complaint_id", "filed_date"]
    ],
    on=vehicle_keys,
    how="left"
)

# Define the 12-month historical window
complaint_windows["history_start"] = (
    complaint_windows["snapshot_date"] -
    pd.DateOffset(months=12)
)

# Use only complaints submitted before snapshot date
complaint_windows["in_history"] = (
    (complaint_windows["filed_date"] >=
     complaint_windows["history_start"]) &
    (complaint_windows["filed_date"] <
     complaint_windows["snapshot_date"])
)

# Count unique complaints in historical window
complaint_windows["valid_complaint_id"] = (
    complaint_windows["complaint_id"].where(
        complaint_windows["in_history"]
    )
)

complaint_counts = (
    complaint_windows
    .groupby(
        vehicle_keys + ["snapshot_date"],
        as_index=False
    )
    .agg(
        complaints_last_12m=(
            "valid_complaint_id", "nunique"
        )
    )
)

# Add feature to modeling grid
feature_grid = feature_grid.merge(
    complaint_counts,
    on=vehicle_keys + ["snapshot_date"],
    how="left",
    validate="one_to_one"
)

feature_grid["complaints_last_12m"] = (
    feature_grid["complaints_last_12m"]
    .fillna(0)
    .astype(int)
)

print("Total samples:", len(feature_grid))

print("\nComplaint count statistics:")
display(feature_grid["complaints_last_12m"].describe())

display(feature_grid.head(10))


Total samples: 2480

Complaint count statistics:


count    2480.000000
mean       48.145968
std        78.165439
min         0.000000
25%         8.000000
50%        22.000000
75%        55.000000
max       888.000000
Name: complaints_last_12m, dtype: float64

,make,model_family,model_year,snapshot_date,future_investigation,complaints_last_12m
0,AUDI,Q5,2015,2016-01-01,0,2
1,AUDI,Q5,2015,2016-04-01,0,2
2,AUDI,Q5,2015,2016-07-01,0,3
3,AUDI,Q5,2015,2016-10-01,0,1
4,AUDI,Q5,2015,2017-01-01,0,1
5,AUDI,Q5,2015,2017-04-01,0,1
6,AUDI,Q5,2015,2017-07-01,0,2
7,AUDI,Q5,2015,2017-10-01,0,5
8,AUDI,Q5,2015,2018-01-01,0,7
9,AUDI,Q5,2015,2018-04-01,0,7


In [48]:

# A5.2 - Build historical safety severity features

import pandas as pd

severity_cols = [
    "crash", "fire",
    "number_of_injuries", "number_of_deaths"
]

# Use only complaints within each historical window
severity_windows = complaint_windows.loc[
    complaint_windows["in_history"]
].copy()

# Attach severity fields from complaint-level dataset
severity_windows = severity_windows.merge(
    complaints_final[
        ["complaint_id"] + severity_cols
    ],
    on="complaint_id",
    how="left",
    validate="many_to_one"
)

# Convert crash/fire indicators to binary
severity_windows["crash_flag"] = (
    severity_windows["crash"] == "Y"
).astype(int)

severity_windows["fire_flag"] = (
    severity_windows["fire"] == "Y"
).astype(int)

# Aggregate by prediction sample
severity_features = (
    severity_windows
    .groupby(
        vehicle_keys + ["snapshot_date"],
        as_index=False
    )
    .agg(
        crash_count_12m=("crash_flag", "sum"),
        fire_count_12m=("fire_flag", "sum"),
        injuries_12m=("number_of_injuries", "sum"),
        deaths_12m=("number_of_deaths", "sum")
    )
)

# Add features to the existing model grid
feature_grid = feature_grid.merge(
    severity_features,
    on=vehicle_keys + ["snapshot_date"],
    how="left",
    validate="one_to_one"
)

count_cols = [
    "crash_count_12m", "fire_count_12m",
    "injuries_12m", "deaths_12m"
]

feature_grid[count_cols] = (
    feature_grid[count_cols].fillna(0)
)

# Rates among complaints in the history window
feature_grid["crash_rate_12m"] = (
    feature_grid["crash_count_12m"] /
    feature_grid["complaints_last_12m"].replace(0, pd.NA)
)

feature_grid["fire_rate_12m"] = (
    feature_grid["fire_count_12m"] /
    feature_grid["complaints_last_12m"].replace(0, pd.NA)
)

print("Total samples:", len(feature_grid))

display(
    feature_grid[
        [
            "complaints_last_12m",
            "crash_count_12m",
            "fire_count_12m",
            "injuries_12m",
            "deaths_12m",
            "crash_rate_12m",
            "fire_rate_12m"
        ]
    ].describe()
)

display(feature_grid.head(10))


Total samples: 2480


,complaints_last_12m,crash_count_12m,fire_count_12m,injuries_12m,deaths_12m
count,2480.000000,2480.000000,2480.000000,2480.000000,2480.000000
mean,48.145968,2.021774,0.484677,1.811290,0.027419
std,78.165439,2.416398,1.081477,2.868342,0.190682
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,8.000000,0.000000,0.000000,0.000000,0.000000
50%,22.000000,1.000000,0.000000,1.000000,0.000000
75%,55.000000,3.000000,1.000000,3.000000,0.000000
max,888.000000,19.000000,18.000000,41.000000,2.000000


,make,model_family,model_year,snapshot_date,future_investigation,complaints_last_12m,crash_count_12m,fire_count_12m,injuries_12m,deaths_12m,crash_rate_12m,fire_rate_12m
0,AUDI,Q5,2015,2016-01-01,0,2,1.0,0.0,1.0,0.0,0.5,0.0
1,AUDI,Q5,2015,2016-04-01,0,2,1.0,0.0,1.0,0.0,0.5,0.0
2,AUDI,Q5,2015,2016-07-01,0,3,1.0,0.0,1.0,0.0,0.333333,0.0
3,AUDI,Q5,2015,2016-10-01,0,1,0.0,0.0,0.0,0.0,0.0,0.0
4,AUDI,Q5,2015,2017-01-01,0,1,0.0,0.0,0.0,0.0,0.0,0.0
5,AUDI,Q5,2015,2017-04-01,0,1,0.0,0.0,0.0,0.0,0.0,0.0
6,AUDI,Q5,2015,2017-07-01,0,2,0.0,0.0,1.0,0.0,0.0,0.0
7,AUDI,Q5,2015,2017-10-01,0,5,0.0,0.0,1.0,0.0,0.0,0.0
8,AUDI,Q5,2015,2018-01-01,0,7,0.0,0.0,1.0,0.0,0.0,0.0
9,AUDI,Q5,2015,2018-04-01,0,7,0.0,0.0,1.0,0.0,0.0,0.0


In [49]:

# A5.3 - Create complaint trend features

import numpy as np
import pandas as pd

trend_windows = complaint_windows.loc[
    complaint_windows["in_history"]
].copy()

# Define 3-month and 6-month boundaries
trend_windows["start_3m"] = (
    trend_windows["snapshot_date"] -
    pd.DateOffset(months=3)
)

trend_windows["start_6m"] = (
    trend_windows["snapshot_date"] -
    pd.DateOffset(months=6)
)

# Count complaints in recent windows
trend_windows["in_3m"] = (
    trend_windows["filed_date"] >=
    trend_windows["start_3m"]
).astype(int)

trend_windows["in_6m"] = (
    trend_windows["filed_date"] >=
    trend_windows["start_6m"]
).astype(int)

trend_features = (
    trend_windows
    .groupby(
        vehicle_keys + ["snapshot_date"],
        as_index=False
    )
    .agg(
        complaints_last_3m=("in_3m", "sum"),
        complaints_last_6m=("in_6m", "sum")
    )
)

# Merge into modeling table
feature_grid = feature_grid.merge(
    trend_features,
    on=vehicle_keys + ["snapshot_date"],
    how="left",
    validate="one_to_one"
)

feature_grid[
    ["complaints_last_3m", "complaints_last_6m"]
] = (
    feature_grid[
        ["complaints_last_3m", "complaints_last_6m"]
    ]
    .fillna(0)
    .astype(int)
)

# Compare recent 3 months with previous 9 months
previous_9m = (
    feature_grid["complaints_last_12m"] -
    feature_grid["complaints_last_3m"]
)

feature_grid["complaint_trend_3m"] = (
    (feature_grid["complaints_last_3m"] / 3) /
    (previous_9m / 9).replace(0, np.nan)
)

# Convert existing rates to numeric
for col in ["crash_rate_12m", "fire_rate_12m"]:
    feature_grid[col] = pd.to_numeric(
        feature_grid[col],
        errors="coerce"
    )

print("Total samples:", len(feature_grid))

display(
    feature_grid[
        [
            "complaints_last_12m",
            "complaints_last_6m",
            "complaints_last_3m",
            "complaint_trend_3m"
        ]
    ].describe()
)

print(
    "Missing trend values:",
    feature_grid["complaint_trend_3m"].isna().sum()
)

assert (
    feature_grid["complaints_last_3m"] <=
    feature_grid["complaints_last_6m"]
).all()

assert (
    feature_grid["complaints_last_6m"] <=
    feature_grid["complaints_last_12m"]
).all()

print("Trend feature validation passed!")


Total samples: 2480


,complaints_last_12m,complaints_last_6m,complaints_last_3m,complaint_trend_3m
count,2480.000000,2480.000000,2480.000000,2305.000000
mean,48.145968,25.491532,13.093548,1.576339
std,78.165439,42.628416,22.402604,2.333397
min,0.000000,0.000000,0.000000,0.000000
25%,8.000000,4.000000,2.000000,0.666667
50%,22.000000,12.000000,6.000000,1.026316
75%,55.000000,28.000000,14.000000,1.593750
max,888.000000,514.000000,260.000000,33.000000


Missing trend values: 175
Trend feature validation passed!


In [50]:

# A5.4 - Create historical component features

import pandas as pd

# Use only complaints from historical 12-month windows
component_windows = complaint_windows.loc[
    complaint_windows["in_history"]
].copy()

# Attach aggregated component descriptions
component_windows = component_windows.merge(
    complaints_final[
        ["complaint_id", "components"]
    ],
    on="complaint_id",
    how="left",
    validate="many_to_one"
)

# Split component strings into distinct components
def parse_components(value):
    if pd.isna(value):
        return set()

    return {
        item.strip().upper()
        for item in str(value).split("|")
        if item.strip()
    }

component_windows["component_set"] = (
    component_windows["components"].apply(parse_components)
)

# Define common component categories
component_keywords = {
    "engine_complaints_12m": "ENGINE",
    "brake_complaints_12m": "BRAKES",
    "electrical_complaints_12m": "ELECTRICAL",
    "powertrain_complaints_12m": "POWER TRAIN",
    "airbag_complaints_12m": "AIR BAGS"
}

# Mark each complaint as involving a component category
for feature_name, keyword in component_keywords.items():
    component_windows[feature_name] = (
        component_windows["component_set"].apply(
            lambda items: int(
                any(keyword in item for item in items)
            )
        )
    )

# Aggregate component counts by snapshot
component_features = (
    component_windows
    .groupby(
        vehicle_keys + ["snapshot_date"],
        as_index=False
    )
    .agg({
        feature_name: "sum"
        for feature_name in component_keywords
    })
)

# Calculate distinct component count within each snapshot
component_diversity = (
    component_windows
    .groupby(vehicle_keys + ["snapshot_date"])["component_set"]
    .apply(
        lambda series: len(
            set().union(*series.tolist())
        )
    )
    .reset_index(name="component_diversity_12m")
)

component_features = component_features.merge(
    component_diversity,
    on=vehicle_keys + ["snapshot_date"],
    how="left",
    validate="one_to_one"
)

# Merge into existing feature table
feature_grid = feature_grid.merge(
    component_features,
    on=vehicle_keys + ["snapshot_date"],
    how="left",
    validate="one_to_one"
)

new_component_cols = (
    list(component_keywords) +
    ["component_diversity_12m"]
)

feature_grid[new_component_cols] = (
    feature_grid[new_component_cols]
    .fillna(0)
    .astype(int)
)

print("Total samples:", len(feature_grid))

display(
    feature_grid[new_component_cols].describe()
)

display(feature_grid.head())


Total samples: 2480


,engine_complaints_12m,brake_complaints_12m,electrical_complaints_12m,powertrain_complaints_12m,airbag_complaints_12m,component_diversity_12m
count,2480.000000,2480.000000,2480.000000,2480.000000,2480.000000,2480.000000
mean,12.317339,3.604435,5.412097,8.692742,2.068952,13.693145
std,40.511878,8.073756,7.352828,21.210920,3.711631,9.365241
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,1.000000,0.000000,0.000000,7.000000
50%,3.000000,1.000000,3.000000,2.000000,1.000000,13.000000
75%,10.000000,3.000000,7.000000,6.000000,3.000000,18.000000
max,675.000000,94.000000,72.000000,285.000000,48.000000,58.000000


,make,model_family,model_year,snapshot_date,future_investigation,complaints_last_12m,crash_count_12m,fire_count_12m,injuries_12m,deaths_12m,...,fire_rate_12m,complaints_last_3m,complaints_last_6m,complaint_trend_3m,engine_complaints_12m,brake_complaints_12m,electrical_complaints_12m,powertrain_complaints_12m,airbag_complaints_12m,component_diversity_12m
0,AUDI,Q5,2015,2016-01-01,0,2,1.0,0.0,1.0,0.0,...,0.0,0,2,0.0,0,0,0,0,0,2
1,AUDI,Q5,2015,2016-04-01,0,2,1.0,0.0,1.0,0.0,...,0.0,0,0,0.0,0,0,0,0,0,2
2,AUDI,Q5,2015,2016-07-01,0,3,1.0,0.0,1.0,0.0,...,0.0,1,1,1.5,0,0,0,0,0,2
3,AUDI,Q5,2015,2016-10-01,0,1,0.0,0.0,0.0,0.0,...,0.0,0,1,0.0,0,0,0,0,0,1
4,AUDI,Q5,2015,2017-01-01,0,1,0.0,0.0,0.0,0.0,...,0.0,0,0,0.0,0,0,0,0,0,1


In [51]:

# A5.5 - Create reporting delay features

import pandas as pd

# Use only complaints in historical 12-month windows
delay_windows = complaint_windows.loc[
    complaint_windows["in_history"]
].copy()

# Attach cleaned reporting delay from complaint-level data
delay_windows = delay_windows.merge(
    complaints_final[
        ["complaint_id", "reporting_delay"]
    ],
    on="complaint_id",
    how="left",
    validate="many_to_one"
)

delay_windows["reporting_delay"] = pd.to_numeric(
    delay_windows["reporting_delay"],
    errors="coerce"
)

# Aggregate reporting delay statistics per prediction sample
delay_features = (
    delay_windows
    .groupby(
        vehicle_keys + ["snapshot_date"],
        as_index=False
    )
    .agg(
        avg_reporting_delay_12m=(
            "reporting_delay", "mean"
        ),
        median_reporting_delay_12m=(
            "reporting_delay", "median"
        ),
        valid_delay_count_12m=(
            "reporting_delay", "count"
        )
    )
)

# Add features to existing modeling table
feature_grid = feature_grid.merge(
    delay_features,
    on=vehicle_keys + ["snapshot_date"],
    how="left",
    validate="one_to_one"
)

# A count of zero is meaningful when there are no valid delays
feature_grid["valid_delay_count_12m"] = (
    feature_grid["valid_delay_count_12m"]
    .fillna(0)
    .astype(int)
)

print("Total samples:", len(feature_grid))

print("\n===== REPORTING DELAY STATISTICS =====")
display(
    feature_grid[
        [
            "avg_reporting_delay_12m",
            "median_reporting_delay_12m",
            "valid_delay_count_12m"
        ]
    ].describe()
)

print("\n===== MISSING VALUES =====")
display(
    feature_grid[
        [
            "avg_reporting_delay_12m",
            "median_reporting_delay_12m"
        ]
    ].isna().sum()
)

# Validation
assert len(feature_grid) == 2480

assert (
    feature_grid["valid_delay_count_12m"] <=
    feature_grid["complaints_last_12m"]
).all()

print("\nReporting delay feature validation passed!")


Total samples: 2480

===== REPORTING DELAY STATISTICS =====


,avg_reporting_delay_12m,median_reporting_delay_12m,valid_delay_count_12m
count,2386.000000,2386.000000,2480.000000
mean,84.264150,20.159891,48.107258
std,67.174300,41.939813,78.131754
min,0.000000,0.000000,0.000000
25%,38.550752,6.000000,8.000000
50%,69.178485,11.000000,22.000000
75%,113.025000,18.000000,55.000000
max,729.000000,729.000000,887.000000



===== MISSING VALUES =====


avg_reporting_delay_12m       94
median_reporting_delay_12m    94
dtype: int64


Reporting delay feature validation passed!


In [52]:

# A5.6 - Validate final modeling features

import numpy as np
import pandas as pd

modeling_data = feature_grid.copy()

sample_keys = [
    "make", "model_family",
    "model_year", "snapshot_date"
]

target_col = "future_investigation"

feature_cols = [
    col for col in modeling_data.columns
    if col not in sample_keys + [target_col]
]

print("===== DATASET SHAPE =====")
print("Total samples:", len(modeling_data))
print("Total columns:", modeling_data.shape[1])
print("Number of features:", len(feature_cols))

print("\n===== UNIQUENESS CHECK =====")
print(
    "Duplicate sample keys:",
    modeling_data.duplicated(sample_keys).sum()
)

print("\n===== TARGET CHECK =====")
display(
    modeling_data[target_col]
    .value_counts()
    .sort_index()
)

print("\n===== FEATURE DATA TYPES =====")
display(modeling_data[feature_cols].dtypes)

print("\n===== MISSING VALUES =====")
display(
    modeling_data[feature_cols]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("\n===== INFINITE VALUES =====")
numeric_features = modeling_data[
    feature_cols
].select_dtypes(include="number")

print(
    "Infinite numeric values:",
    np.isinf(
        numeric_features.to_numpy(dtype=float)
    ).sum()
)

# Validate key properties
assert len(modeling_data) == 2480
assert not modeling_data.duplicated(sample_keys).any()
assert modeling_data[target_col].isin([0, 1]).all()

print("\nBasic validation passed!")


===== DATASET SHAPE =====
Total samples: 2480
Total columns: 24
Number of features: 19

===== UNIQUENESS CHECK =====
Duplicate sample keys: 0

===== TARGET CHECK =====


future_investigation
0    2123
1     357
Name: count, dtype: int64


===== FEATURE DATA TYPES =====


complaints_last_12m             int64
crash_count_12m               float64
fire_count_12m                float64
injuries_12m                  float64
deaths_12m                    float64
crash_rate_12m                float64
fire_rate_12m                 float64
complaints_last_3m              int64
complaints_last_6m              int64
complaint_trend_3m            float64
engine_complaints_12m           int64
brake_complaints_12m            int64
electrical_complaints_12m       int64
powertrain_complaints_12m       int64
airbag_complaints_12m           int64
component_diversity_12m         int64
avg_reporting_delay_12m       float64
median_reporting_delay_12m    float64
valid_delay_count_12m           int64
dtype: object


===== MISSING VALUES =====


complaint_trend_3m            175
median_reporting_delay_12m     94
avg_reporting_delay_12m        94
crash_rate_12m                 93
fire_rate_12m                  93
brake_complaints_12m            0
component_diversity_12m         0
airbag_complaints_12m           0
powertrain_complaints_12m       0
electrical_complaints_12m       0
complaints_last_12m             0
engine_complaints_12m           0
crash_count_12m                 0
complaints_last_6m              0
complaints_last_3m              0
deaths_12m                      0
injuries_12m                    0
fire_count_12m                  0
valid_delay_count_12m           0
dtype: int64


===== INFINITE VALUES =====
Infinite numeric values: 0

Basic validation passed!


In [53]:

# A5.7 - Save final modeling dataset

from pathlib import Path

# Project directory
project_dir = Path(
    ROOT
)

output_dir = project_dir / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "modeling_dataset.csv"

# Keep samples in chronological order
modeling_data = modeling_data.sort_values(
    ["snapshot_date", "make", "model_family", "model_year"]
).reset_index(drop=True)

modeling_data.to_csv(output_path, index=False)

print("Dataset saved to:", output_path)
print("Shape:", modeling_data.shape)

# Reload and verify
check_data = pd.read_csv(
    output_path,
    parse_dates=["snapshot_date"]
)

assert check_data.shape == modeling_data.shape
assert check_data["future_investigation"].sum() == 357

print("CSV reload validation passed!")


Dataset saved to: /Users/xinyisong/Desktop/DATA1030-Midterm-Xinfinity/data/processed/modeling_dataset.csv
Shape: (2480, 24)
CSV reload validation passed!
